# Retrieval Corpus Preparation (No Review Comments, No Synthetic Data)

This notebook creates retrieval corpus chunks from:
- project_guideline (repo-specific)
- pep8 (common)
- pep257 (common)
- linter_rule (common)

It does NOT use review comments, LLM-generated text, or synthetic augmentation.

Designed for team parallelization: each person runs one repo and exports one partial JSON.

In [ ]:
# Optional install (run once)
# !pip install -q requests beautifulsoup4 lxml

In [ ]:
from pathlib import Path
import json
import re
import requests
from bs4 import BeautifulSoup

ROOT = Path('..').resolve()
OUT_DIR = ROOT / 'data' / 'raw' / 'dataset_v3'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR = OUT_DIR / 'partials_retrieval'
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

ALL_REPOS = [
    'django/django',
    'pandas-dev/pandas',
    'scikit-learn/scikit-learn',
    'pallets/flask',
    'fastapi/fastapi',
]

# Choose ONE repo per person (comment/uncomment)
TARGET_REPO = 'django/django'
# TARGET_REPO = 'pandas-dev/pandas'
# TARGET_REPO = 'scikit-learn/scikit-learn'
# TARGET_REPO = 'pallets/flask'
# TARGET_REPO = 'fastapi/fastapi'

CATEGORIES = [
    'indentation',
    'naming_convention',
    'unused_import',
    'mutable_default',
    'documentation_formatting',
]

print('Target repo:', TARGET_REPO)
print('Output directory:', OUT_DIR)

In [ ]:
# Optional GitHub token for private-rate-limit bump (public files often work without token)
GITHUB_TOKEN = ''  # paste token if needed
HEADERS = {'Accept': 'application/vnd.github+json'}
if GITHUB_TOKEN.strip():
    HEADERS['Authorization'] = f'Bearer {GITHUB_TOKEN.strip()}'

In [ ]:
GUIDE_PATHS = [
    'CONTRIBUTING.md',
    'CONTRIBUTING.rst',
    '.github/CONTRIBUTING.md',
    'docs/contributing.md',
    'docs/contributing.rst',
    'docs/development/contributing.rst',
    'docs/internals/contributing/writing-code/coding-style.txt',
]

GUIDE_CATEGORY_KEYWORDS = {
    'indentation': ['indent', 'whitespace', 'spaces', 'tabs', 'line length', 'wrap', 'alignment'],
    'naming_convention': ['naming', 'name', 'snake_case', 'camelcase', 'pascalcase', 'convention'],
    'unused_import': ['unused import', 'remove import', 'import'],
    'mutable_default': ['mutable', 'default argument', 'default parameter', 'none sentinel'],
    'documentation_formatting': ['docstring', 'documentation', 'comment style', 'pep 257', 'formatting'],
}

def github_get_text(repo: str, path: str) -> str:
    url = f'https://api.github.com/repos/{repo}/contents/{path}'
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        return ''
    payload = r.json()
    dl = payload.get('download_url')
    if not dl:
        return ''
    rr = requests.get(dl, timeout=30)
    if rr.status_code != 200:
        return ''
    return rr.text

def chunk_words(text: str, chunk_size: int = 220) -> list[str]:
    words = text.split()
    out = []
    for i in range(0, len(words), chunk_size):
        c = ' '.join(words[i:i + chunk_size]).strip()
        if len(c) >= 60:
            out.append(c)
    return out

def classify_guideline_paragraph(paragraph: str) -> str | None:
    p = paragraph.lower()
    for cat, kws in GUIDE_CATEGORY_KEYWORDS.items():
        if any(k in p for k in kws):
            return cat
    return None

def build_project_guidelines(repo: str) -> list[dict]:
    chunks = []
    for p in GUIDE_PATHS:
        txt = github_get_text(repo, p)
        if not txt.strip():
            continue
        paragraphs = [x.strip() for x in re.split(r'\n\n+', txt) if x.strip()]
        for para in paragraphs:
            cat = classify_guideline_paragraph(para)
            if not cat:
                continue
            for c in chunk_words(para, chunk_size=220):
                chunks.append({
                    'text': c,
                    'category': cat,
                    'source_type': 'project_guideline',
                    'repo': repo,
                    'source_doc': p,
                })
    return chunks

In [ ]:
PEP8_SECTION_MAP = {
    'indentation': 'indentation',
    'tabs or spaces': 'indentation',
    'maximum line length': 'indentation',
    'blank lines': 'indentation',
    'whitespace in expressions and statements': 'indentation',
    'imports': 'unused_import',
    'naming conventions': 'naming_convention',
    'function and variable names': 'naming_convention',
    'class names': 'naming_convention',
    'programming recommendations': 'mutable_default',
    'documentation strings': 'documentation_formatting',
    'comments': 'documentation_formatting',
}

PEP257_SECTION_MAP = {
    'what is a docstring': 'documentation_formatting',
    'one-line docstrings': 'documentation_formatting',
    'multi-line docstrings': 'documentation_formatting',
    'handling docstring indentation': 'documentation_formatting',
}

def scrape_pep(url: str, section_map: dict[str, str], source_type: str) -> list[dict]:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'lxml')
    main = soup.find('article') or soup.find('main') or soup.body
    chunks = []
    current_title = ''
    current_text = []

    for el in main.find_all(['h2', 'h3', 'p', 'pre']):
        if el.name in ('h2', 'h3'):
            if current_title and current_text:
                cat = section_map.get(current_title.lower().strip())
                if cat:
                    text = current_title + '\n\n' + '\n'.join(current_text)
                    for c in chunk_words(text, chunk_size=240):
                        chunks.append({
                            'text': c,
                            'category': cat,
                            'source_type': source_type,
                            'repo': 'common',
                            'source_doc': url,
                        })
            current_title = el.get_text().strip()
            current_text = []
        else:
            t = el.get_text().strip()
            if t:
                current_text.append(t)

    if current_title and current_text:
        cat = section_map.get(current_title.lower().strip())
        if cat:
            text = current_title + '\n\n' + '\n'.join(current_text)
            for c in chunk_words(text, chunk_size=240):
                chunks.append({
                    'text': c,
                    'category': cat,
                    'source_type': source_type,
                    'repo': 'common',
                    'source_doc': url,
                })

    return chunks

LINTER_RULES = [
    {'text': 'E111/W191 and related indentation rules: use consistent 4-space indentation and avoid tab-based indentation.', 'category': 'indentation', 'source_type': 'linter_rule', 'repo': 'common', 'source_doc': 'flake8/pylint'},
    {'text': 'E501 and line wrap guidance: keep line length within style constraints and use clean continuation.', 'category': 'indentation', 'source_type': 'linter_rule', 'repo': 'common', 'source_doc': 'flake8'},
    {'text': 'C0103 and pep8-naming rules: use snake_case for functions/variables and CapWords for classes.', 'category': 'naming_convention', 'source_type': 'linter_rule', 'repo': 'common', 'source_doc': 'pylint/pep8-naming'},
    {'text': 'F401/W0611: remove imports that are not used.', 'category': 'unused_import', 'source_type': 'linter_rule', 'repo': 'common', 'source_doc': 'flake8/pylint'},
    {'text': 'W0102/B006: avoid mutable default argument values; use None sentinel and initialize in function body.', 'category': 'mutable_default', 'source_type': 'linter_rule', 'repo': 'common', 'source_doc': 'pylint/flake8-bugbear'},
    {'text': 'D100-D107, D200 and related docstring rules: provide clear module/class/function docstrings with consistent formatting.', 'category': 'documentation_formatting', 'source_type': 'linter_rule', 'repo': 'common', 'source_doc': 'pydocstyle'},
]

In [ ]:
repo_guidelines = build_project_guidelines(TARGET_REPO)
pep8_chunks = scrape_pep('https://peps.python.org/pep-0008/', PEP8_SECTION_MAP, 'pep8')
pep257_chunks = scrape_pep('https://peps.python.org/pep-0257/', PEP257_SECTION_MAP, 'pep257')

partial = repo_guidelines + pep8_chunks + pep257_chunks + LINTER_RULES

# de-dup by normalized text prefix
seen = set()
dedup = []
for e in partial:
    k = re.sub(r'\s+', ' ', e['text']).strip().lower()[:220]
    if k in seen:
        continue
    seen.add(k)
    dedup.append(e)

for i, e in enumerate(dedup, start=1):
    slug = TARGET_REPO.replace('/', '_')
    e['chunk_id'] = f'{slug}_chunk_{i:04d}'

partial_path = PARTIAL_DIR / f'retrieval_partial_{TARGET_REPO.replace('/', '_')}.json'
with partial_path.open('w', encoding='utf-8') as f:
    json.dump(dedup, f, indent=2, ensure_ascii=False)

print('Saved partial corpus:', partial_path)
print('Total chunks:', len(dedup))

In [ ]:
# Run this after all teammates generated their partial files
partials = sorted(PARTIAL_DIR.glob('retrieval_partial_*.json'))
all_entries = []
seen = set()
for p in partials:
    with p.open('r', encoding='utf-8') as f:
        arr = json.load(f)
    for e in arr:
        k = re.sub(r'\s+', ' ', e['text']).strip().lower()[:220]
        if k in seen:
            continue
        seen.add(k)
        all_entries.append(e)

for i, e in enumerate(all_entries, start=1):
    e['chunk_id'] = f'chunk_{i:04d}'

final_path = OUT_DIR / 'retrieval_corpus.json'
with final_path.open('w', encoding='utf-8') as f:
    json.dump(all_entries, f, indent=2, ensure_ascii=False)

print('Merged partial files:', len(partials))
print('Final corpus size:', len(all_entries))
print('Saved:', final_path)